In [1]:
import os
import random
import numpy as np
import h5py
import matplotlib.pyplot as plt
import open3d as o3d
import plotly.graph_objects as go

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# Train/Valid Datasets

In [6]:
# CHANGRABLE VARIABLES
DATASET_DIR = r"D:\Datasets\MinimarketPointCloud\MiniMarket_point_clouds\2048\segmentation_dataset\FLOORED_ketchup_heinz_400ml_segmentation_20250526_121710_numPoints_2048_maxObjects_10_orientations_1.h5"

In [7]:
def get_point_cloud(DATASET_DIR, N):
    with h5py.File(DATASET_DIR, 'r') as f:
        # Read datasets
        seg_points = f["seg_points"][:]  
        seg_colors = f["seg_colors"][:]  
        seg_labels = f["seg_labels"][:]  
    print(seg_points.shape)
    print(seg_colors.shape)
    print(seg_labels.shape)
    
    print("Random sample index:", N)
    points_sample = seg_points[N, :, :]
    colors_sample = seg_colors[N, :, :]
    labels_sample = np.where((seg_labels[N, :, :] == np.array([1, 0])).all(axis=1, keepdims=True), [0, 1, 0], [1, 0, 0])
    return points_sample, colors_sample, labels_sample

def matplotlib_pc(points_sample, colors_sample, labels_sample):

    fig = plt.figure(figsize=(12, 6))

    # First subplot for color visualization
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.scatter(points_sample[:, 0], points_sample[:, 1], points_sample[:, 2],
                c=colors_sample, s=1)
    ax1.set_title("Color Visualization")
    ax1.set_axis_off()

    # Second subplot for label visualization
    ax2 = fig.add_subplot(122, projection='3d')
    ax2.scatter(points_sample[:, 0], points_sample[:, 1], points_sample[:, 2],
                c=labels_sample, s=1)
    ax2.set_title("Label Visualization")
    ax2.set_axis_off()

    plt.show()

def open3d_pc(points_sample, colors_sample, labels_sample):
    pcd_colors = o3d.geometry.PointCloud()
    pcd_colors.points = o3d.utility.Vector3dVector(points_sample)
    pcd_colors.colors = o3d.utility.Vector3dVector(colors_sample)
    o3d.visualization.draw_geometries([pcd_colors], window_name="Color Visualization")

    pcd_labels = o3d.geometry.PointCloud()
    pcd_labels.points = o3d.utility.Vector3dVector(points_sample)
    pcd_labels.colors = o3d.utility.Vector3dVector(labels_sample)
    o3d.visualization.draw_geometries([pcd_labels], window_name="Label Visualization")



In [8]:
points_sample, colors_sample, labels_sample = get_point_cloud(DATASET_DIR, 1)
matplotlib_pc(points_sample, colors_sample, labels_sample)
# open3d_pc(points_sample, colors_sample, labels_sample)

(1, 22480, 3)
(1, 22480, 3)
(1, 22480, 2)
Random sample index: 1


IndexError: index 1 is out of bounds for axis 0 with size 1

# Test Datasets

In [2]:
# CHANGRABLE VARIABLES
dataset_name = 'all_test_scenes.h5'
N = 2  # Change this to visualize different scenes
DATASET_DIR = os.path.join(os.getcwd(), dataset_name)

In [3]:
with h5py.File(DATASET_DIR, 'r') as f:
    # Read datasets
    point_clouds = f["point_clouds"][:]
    color_clouds = f["color_clouds"][:]
    print("f keys:", f.keys())

f keys: <KeysViewHDF5 ['color_clouds', 'point_clouds']>


In [2]:
"print this"

'print this'

In [3]:
def get_point_cloud(DATASET_DIR, N, percentage=1):
    with h5py.File(DATASET_DIR, 'r') as f:
        # Read datasets
        point_clouds = f["point_clouds"][:]
        color_clouds = f["color_clouds"][:]
    B, num_points, _ = point_clouds.shape
    print("Number of clouds:", B, ", Chosen cloud index:", N)
    new_num_points = int(num_points * percentage)
    idx_downsample = random.sample(range(num_points), new_num_points)
    point_clouds = point_clouds[N, idx_downsample, :]
    color_clouds = color_clouds[N, idx_downsample, :]
    valid_mask = ~np.isnan(point_clouds).any(axis=1) & ~np.isnan(color_clouds).any(axis=1)
    point_clouds = point_clouds[valid_mask]
    color_clouds = color_clouds[valid_mask]
    return point_clouds, color_clouds

def plot_pc(point_clouds, color_clouds):
    fig = go.Figure(data=[go.Scatter3d(
        x=point_clouds[:, 0],
        y=point_clouds[:, 1],
        z=point_clouds[:, 2],
        mode='markers',
        marker=dict(
            size=1,
            color=color_clouds,  # Can be RGB or scalar values
            opacity=0.8
        )
    )])

    fig.update_layout(
        title="Interactive 3D Point Cloud",
        scene=dict(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False),
        margin=dict(l=0, r=0, b=0, t=30)
    )

    fig.show()



In [4]:
point_clouds, color_clouds = get_point_cloud(DATASET_DIR, N,0.3)
# plot_pc(point_clouds, color_clouds)

Number of clouds: 1 , Chosen cloud index: 2


IndexError: index 2 is out of bounds for axis 0 with size 1

In [8]:
point_clouds

array([[-0.2509602 ,  0.18914998,  0.76700002],
       [-0.08639237, -0.04367303,  0.80800003],
       [ 0.10618373,  0.220488  ,  0.77300006],
       ...,
       [-0.29625863,  0.18712144,  0.76200002],
       [-0.14866179,  0.19821598,  0.76800001],
       [ 0.1067481 , -0.0918696 ,  0.84800005]])